# Generates report of all binding failures

In [ ]:
# %% Export All Binding Failures to Text File

import json
from pathlib import Path
from datetime import datetime

# Configuration
TYPE = 'custom'  # Change to 'athletes' if needed
BASE_DIR = f"./binding_failures_{TYPE}_temp0.3"
BASE_JSON = f'binding_test_cases_{TYPE}.json'
OUTPUT_FILE = f"all_binding_failures_{TYPE}.txt"
BIND_THRESH = 0.6  # Threshold for considering a binding as failed

BINDING_TYPES = [
    "binding_raw",
    "binding_reason", 
    "binding_no_reason",
    "binding_mc_raw",
    "binding_mc_reason",
    "binding_mc_no_reason",
]

def parse_list_from_string(s):
    """Parse a list from string representation, handling various formats."""
    import ast
    if isinstance(s, list):
        return s
    if isinstance(s, str):
        try:
            # Try to parse as Python literal
            return ast.literal_eval(s)
        except:
            # Try to extract JSON-like list from text
            import re
            match = re.search(r'\[.*?\]', s)
            if match:
                try:
                    return ast.literal_eval(match.group(0))
                except:
                    pass
    return []

def compare_answers(response, expected):
    """Compare model response to expected answer, return True if they match."""
    # Parse both as lists
    response_list = parse_list_from_string(response)
    expected_list = parse_list_from_string(expected)
    
    # Sort and compare
    return sorted([str(x).strip().lower() for x in response_list]) == \
           sorted([str(x).strip().lower() for x in expected_list])

print(f"Configuration:")
print(f"  TYPE: {TYPE}")
print(f"  BASE_DIR: {BASE_DIR}")
print(f"  BASE_JSON: {BASE_JSON}")
print(f"  OUTPUT_FILE: {OUTPUT_FILE}")
print(f"  BIND_THRESH: {BIND_THRESH}")

base_path = Path(BASE_DIR)

if not base_path.exists():
    print(f"\n❌ ERROR: Base directory not found: {BASE_DIR}")
else:
    # Load JSON entities to filter
    json_entities = set()  # Will store "fail_entity_name" format
    if Path(BASE_JSON).exists():
        with open(BASE_JSON, 'r') as f:
            data = json.load(f)
        for entity_name in data['test_cases'].keys():
            # Convert: "Some Entity Name" -> "fail_some_entity_name"
            dir_name = f"fail_{entity_name.replace(' ', '_').lower()}"
            json_entities.add(dir_name)
        print(f"\n✅ Loaded {len(json_entities)} entities from {BASE_JSON}")
    else:
        print(f"\n⚠️  WARNING: JSON file not found: {BASE_JSON}")
        print(f"   Will process ALL entities in directories")
    
    # Get all models
    models = sorted([d.name for d in base_path.iterdir() if d.is_dir()])
    
    print(f"\nFound {len(models)} models")
    print(f"Binding types to check: {BINDING_TYPES}")
    print(f"\nProcessing...\n")
    
    total_failures = 0
    model_stats = {}
    
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as out:
        # Write header
        out.write("="*100 + "\n")
        out.write("BINDING FAILURES - ALL MODELS AND TYPES\n")
        out.write(f"Dataset: {TYPE}\n")
        out.write(f"Threshold: {BIND_THRESH}\n")
        out.write(f"JSON Filter: {BASE_JSON} ({len(json_entities)} entities)\n" if json_entities else "JSON Filter: None (processing all entities)\n")
        out.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        out.write("="*100 + "\n\n")
        
        # Process each model
        for model_idx, model_name in enumerate(models, 1):
            model_path = base_path / model_name
            
            print(f"[{model_idx}/{len(models)}] Processing: {model_name}")
            
            model_failures = 0
            binding_type_stats = {bt: 0 for bt in BINDING_TYPES}
            
            out.write("\n" + "="*100 + "\n")
            out.write(f"MODEL: {model_name}\n")
            out.write("="*100 + "\n\n")
            
            # Process each entity (fail_* directory)
            fail_dirs = sorted(model_path.glob("fail_*"))
            
            for fail_dir in fail_dirs:
                entity_name = fail_dir.name
                
                # Skip if not in JSON file (when JSON file is provided)
                if json_entities and entity_name not in json_entities:
                    continue
                
                # Check each binding type
                for binding_type in BINDING_TYPES:
                    eval_dir = fail_dir / f"eval_{binding_type}"
                    
                    if not eval_dir.exists():
                        continue
                    
                    # Check all evaluation files to compute accuracy
                    eval_files = sorted([f for f in eval_dir.glob("eval_*.json") 
                                       if not f.name.startswith("_debug")])
                    
                    if not eval_files:
                        continue
                    
                    # Compute accuracy across all eval files
                    correct_count = 0
                    total_count = 0
                    all_data = []
                    
                    for eval_file in eval_files:
                        try:
                            with open(eval_file, 'r', encoding='utf-8') as f:
                                data = json.load(f)
                            
                            total_count += 1
                            
                            # Check correctness using both evaluation flag AND direct comparison
                            is_correct_flag = data.get('correct', False) or \
                                            data.get('evaluation', {}).get('correct', False)
                            
                            # Also check by comparing response to expected
                            response = data.get('response', '')
                            expected = data.get('expected', '')
                            # is_correct_comparison = compare_answers(response, expected)
                            
                            # Use the stricter criterion (if either says incorrect, it's incorrect)
                            is_correct = is_correct_flag # and is_correct_comparison
                            
                            if is_correct:
                                correct_count += 1
                            
                            all_data.append((eval_file, data, is_correct))
                        
                        except Exception as e:
                            print(f"  ⚠️  Error reading {eval_file}: {e}")
                            continue
                    
                    # Calculate accuracy
                    if total_count == 0:
                        continue
                    
                    accuracy = correct_count / total_count
                    
                    # Check if this is a binding failure based on threshold
                    if accuracy < BIND_THRESH:
                        # This is a binding failure - save ONE example
                        model_failures += 1
                        total_failures += 1
                        binding_type_stats[binding_type] += 1
                        
                        # Find the first incorrect example to display
                        example_data = None
                        example_file = None
                        
                        for eval_file, data, is_correct in all_data:
                            if not is_correct:
                                example_data = data
                                example_file = eval_file
                                break
                        
                        if example_data is None:
                            continue
                        
                        # Try to load the original prompt from bind_{type}/{id}.json
                        original_prompt = None
                        eval_id = example_file.stem.split('_')[-1]  # Extract ID from eval_X.json
                        # binding_type_short = binding_type.replace('binding_', '')  # e.g., "reason"
                        original_file = fail_dir / binding_type / f"{eval_id}.json"
                        
                        if original_file.exists():
                            try:
                                with open(original_file, 'r', encoding='utf-8') as f:
                                    original_data = json.load(f)
                                    original_prompt = original_data.get('prompt', None)
                            except Exception as e:
                                pass
                        
                        # Write to file
                        out.write("-"*100 + "\n")
                        out.write(f"FAILURE #{model_failures} | Entity: {entity_name} | "
                                f"Type: {binding_type} | Accuracy: {accuracy:.2f} ({correct_count}/{total_count})\n")
                        out.write(f"Example from: {example_file.name}\n")
                        out.write("-"*100 + "\n\n")
                        
                        # Write original query/prompt (from bind_{type}/{id}.json)
                        if original_prompt:
                            out.write("📝 ORIGINAL QUERY/PROMPT:\n")
                            out.write(original_prompt + "\n\n")
                        
                        # Write evaluation query (from eval_{type}/eval_{id}.json) if different
                        if 'query' in example_data and example_data['query'] != original_prompt:
                            out.write("🔍 EVALUATION QUERY:\n")
                            out.write(example_data['query'] + "\n\n")
                        
                        # Write model response
                        if 'response' in example_data:
                            out.write("🤖 MODEL RESPONSE:\n")
                            out.write(str(example_data['response']) + "\n\n")
                        
                        # Write expected answer
                        if 'expected' in example_data:
                            out.write("✓ EXPECTED ANSWER:\n")
                            out.write(str(example_data['expected']) + "\n\n")
                        
                        # Write evaluation reasoning if available
                        if 'reasoning_trace' in original_data:
                            out.write("💭 EVALUATION REASONING:\n")
                            out.write(original_data['reasoning_trace'] + "\n\n")
                        elif 'evaluation' in original_data and 'reasoning' in original_data:
                            out.write("💭 EVALUATION REASONING:\n")
                            out.write(original_data['reasoning_trace'] + "\n\n")
                        
                        # Write evaluation details
                        if 'evaluation' in example_data:
                            out.write("📊 EVALUATION DETAILS:\n")
                            out.write(f"  Found expected: {example_data['evaluation'].get('found_expected', [])}\n")
                            out.write(f"  Missing: {example_data['evaluation'].get('missing', [])}\n")
                            out.write(f"  Eval correct flag: {example_data['evaluation'].get('correct', False)}\n\n")
                        
                        # Write correct flag
                        out.write(f"❌ CORRECT: False\n")
                        out.write("\n" + "="*100 + "\n\n")
            
            # Write model summary
            out.write("\n" + "-"*100 + "\n")
            out.write(f"SUMMARY FOR {model_name}:\n")
            out.write(f"Total Failures: {model_failures}\n")
            out.write("Failures by Type:\n")
            for bt, count in binding_type_stats.items():
                if count > 0:
                    out.write(f"  - {bt}: {count}\n")
            out.write("-"*100 + "\n\n")
            
            # Store stats
            model_stats[model_name] = {
                'total': model_failures,
                'by_type': binding_type_stats
            }
            
            print(f"  → Found {model_failures} failures (accuracy < {BIND_THRESH})")
        
        # Write global summary at the end
        out.write("\n\n" + "="*100 + "\n")
        out.write("GLOBAL SUMMARY\n")
        out.write("="*100 + "\n\n")
        out.write(f"Total Models: {len(models)}\n")
        out.write(f"Total Failures Across All Models: {total_failures}\n\n")
        out.write("Failures per Model:\n")
        for model_name, stats in model_stats.items():
            out.write(f"\n{model_name}: {stats['total']} failures\n")
            for bt, count in stats['by_type'].items():
                if count > 0:
                    out.write(f"  - {bt}: {count}\n")
        out.write("\n" + "="*100 + "\n")
    
    print(f"\n{'='*80}")
    print(f"✅ Export complete!")
    print(f"Total failures exported: {total_failures}")
    print(f"Output file: {OUTPUT_FILE}")
    print(f"{'='*80}\n")
    
    # Print summary table
    print("Summary by Model:")
    print("-" * 60)
    for model_name, stats in model_stats.items():
        print(f"{model_name:30s} {stats['total']:5d} failures")
        for bt, count in stats['by_type'].items():
            if count > 0:
                print(f"  └─ {bt:30s} {count:5d}")
    print("-" * 60)

In [ ]:
from pathlib import Path
import json

TYPE = 'athletes'
# TYPE = 'custom'

BASE_DIR = f"./binding_failures_{TYPE}_temp0.3"
BASE_JSON = f'binding_test_cases_{TYPE}.json'

# BASE_DIR = f"binding_failures_custom_strict_very_temp0.3"
# BASE_JSON = f'binding_test_cases_custom_strict_very.json'

# load reference JSON (examples must exist here)
with open(BASE_JSON) as f:
    ref = json.load(f)

valid_examples = set(
    e.replace(" ", "_").lower()
    for e in ref["test_cases"].keys()
)

counts = {}

for model_dir in Path(BASE_DIR).iterdir():
    if not model_dir.is_dir():
        continue

    for fail_dir in model_dir.glob("fail_*"):
        entity = fail_dir.name[5:]  # remove "fail_"
        if entity not in valid_examples:
            continue  # example not in JSON

        for eval_dir in fail_dir.glob("eval_*"):
            if not eval_dir.is_dir():
                continue

            n_evals = sum(
                1 for f in eval_dir.glob("eval_*.json")
                if not f.name.startswith("_debug")
            )

            if n_evals < 5:
                counts.setdefault(model_dir.name, {})[
                    f"{fail_dir.name}/{eval_dir.name}"
                ] = n_evals

if len(counts) == 0:
    print("No Entity with missing evaluation file, nor binding experiment")

print("Missing binding: ")
# print results
for model, v in counts.items():
    print(f"\n{model}")
    for k, n in sorted(v.items()):
        print(f"  {k}: {n}")


# Countin Failures

### Athletes

In [ ]:
"""
Binding Failures Evaluation - All Models Comparison
====================================================
"""

import os
import json
import glob
import re
from pathlib import Path
import pandas as pd
from typing import Dict, List, Tuple, Set


TYPE = 'athletes'
BASE_DIR = f"./binding_failures_{TYPE}_temp0.3"
DATASET_JSON = f'binding_test_cases_{TYPE}.json'
TEMPERATURE = 0.3
FACT_KNOW_THRESH = 0.6
BIND_THRESH = 0.6 # Correct when 60% of eval_bind are correct.

BINDING_TYPES = [
    "binding_raw",
    "binding_reason", 
    "binding_no_reason",
    "binding_mc_raw",
    "binding_mc_reason",
    "binding_mc_no_reason",
]

# %% [markdown]
# ## Helper Functions

# %%
def _compute_accuracy(eval_dir: Path, pattern: str = "eval_*.json") -> float:
    correct, total = 0, 0
    for f in eval_dir.glob(pattern):
        if f.name.startswith("_debug"):
            continue
        data = json.load(open(f))
        total += 1
        is_correct = data.get('correct', False) or data.get('evaluation', {}).get('correct', False)
        if is_correct:
            correct += 1
    return correct / total if total > 0 else float('nan')


def count_binding_failures(model_name: str, temperature: float = 0.3, 
                           base_dir: str = "experiments/Binding_Failures/binding_failures",
                           json_path: str = None,
                           fact_know_thresh: float = None, 
                           bind_thresh: float = None,
                           show_entity_names: bool = False,
                           filter_to_json_only: bool = True):
    """
    Build a table of fact/binding accuracies per fail, and optionally identify binding failures.
    
    Args:
        filter_to_json_only: If True, only include entities that exist in the JSON file.
    
    Returns:
        If fact_know_thresh and bind_thresh are None:
            df, json_entities, evaluated_entities
        Otherwise:
            df, binding_failures, json_entities, evaluated_entities
    
    Note: json_entities are in "fail_{name}" format to match directory names.
    """
    model_path = Path(base_dir) / model_name.replace(':', '_')
    
    if not model_path.exists():
        raise FileNotFoundError(f"Model path not found: {model_path}")
    
    # Load JSON entities for comparison
    # Entity names in JSON are converted to directory format: fail_{entity.replace(' ', '_').lower()}
    json_entities: Set[str] = set()  # Will store full "fail_*" names
    json_entity_parts: Set[str] = set()  # Will store just the entity part (without "fail_" prefix)
    entity_to_id = {}
    if json_path and os.path.exists(json_path):
        with open(json_path, 'r') as f:
            data = json.load(f)
        for idx, entity_name in enumerate(data['test_cases'].keys(), start=1):
            # Convert: "Some Entity Name" -> "some_entity_name"
            dir_name = entity_name.replace(' ', '_').lower()
            entity_to_id[dir_name] = idx
            json_entity_parts.add(dir_name)
            json_entities.add(f"fail_{dir_name}")  # Full directory name format
    
    results = {}
    fail_dirs = list(model_path.glob("fail_*"))
    
    # Track evaluated entities (the raw entity part, not the display name)
    evaluated_entities: Set[str] = set()
    
    name_mapping = {}
    entity_part_mapping = {}  # Maps display_name back to entity_part
    
    # Track which directories are excluded because they're not in JSON
    excluded_dirs = []
    
    for fail_dir in fail_dirs:
        original_name = fail_dir.name
        entity_part = original_name[5:]  # Remove "fail_" prefix
        
        # Skip entities not in JSON if filtering is enabled
        if filter_to_json_only and json_path and original_name not in json_entities:
            excluded_dirs.append(original_name)
            continue
        
        evaluated_entities.add(entity_part)
        
        if json_path and entity_part in entity_to_id:
            display_name = f"fail_{entity_to_id[entity_part]}" if not show_entity_names else original_name
        elif re.match(r'^\d+$', entity_part):
            display_name = original_name
        else:
            display_name = original_name if show_entity_names else original_name
        
        name_mapping[original_name] = display_name
        entity_part_mapping[display_name] = entity_part
    
    # Store excluded directories info for later reporting
    excluded_info = {
        'count': len(excluded_dirs),
        'dirs': excluded_dirs
    }
    
    for fail_dir in fail_dirs:
        original_name = fail_dir.name
        
        # Skip if not in name_mapping (means it was excluded)
        if original_name not in name_mapping:
            continue
            
        display_name = name_mapping[original_name]
        results[display_name] = {}
        
        facts = set()
        for subdir in fail_dir.iterdir():
            if subdir.is_dir() and subdir.name.startswith("eval_"):
                name = subdir.name
                if re.match(r'^eval_[a-z]$', name):
                    facts.add(name[5:])
                elif re.match(r'^eval_\w+_[a-z]$', name):
                    facts.add(name.split('_')[-1])
        
        facts = sorted(facts)
        
        for fact in facts:
            for eval_dir in fail_dir.glob(f"eval_*_{fact}"):
                qtype = eval_dir.name.replace(f"_{fact}", "").replace("eval_", "")
                results[display_name][f"{qtype}_{fact}"] = _compute_accuracy(eval_dir, pattern="eval_*.json")
            
            yesno_dir = fail_dir / f"eval_{fact}"
            if yesno_dir.exists():
                results[display_name][f"yesno_{fact}"] = _compute_accuracy(yesno_dir, pattern="eval_*.json")
        
        for binding_type in BINDING_TYPES:
            binding_dir = fail_dir / f"eval_{binding_type}"
            if binding_dir.exists():
                results[display_name][binding_type] = _compute_accuracy(
                    binding_dir, pattern="eval_*.json"
                )
    
    df = pd.DataFrame.from_dict(results, orient='index')
    
    # Store the entity_part_mapping in df attrs for later use
    # Also store reverse mapping: display_name -> original "fail_*" name
    df.attrs['entity_part_mapping'] = entity_part_mapping
    df.attrs['name_mapping'] = name_mapping  # original_name -> display_name
    df.attrs['reverse_name_mapping'] = {v: k for k, v in name_mapping.items()}  # display_name -> original_name
    df.attrs['excluded_info'] = excluded_info  # Info about excluded directories
    
    fact_cols = []
    all_facts = set()
    for col in df.columns:
        if col not in BINDING_TYPES:
            match = re.search(r'_([a-z])$', col)
            if match:
                all_facts.add(match.group(1))
    
    all_prefixes = sorted(set(col.rsplit('_', 1)[0] for col in df.columns if col not in BINDING_TYPES))
    for fact in sorted(all_facts):
        for prefix in all_prefixes:
            col = f"{prefix}_{fact}"
            if col in df.columns:
                fact_cols.append(col)
    
    for binding_type in BINDING_TYPES:
        if binding_type in df.columns:
            fact_cols.append(binding_type)
    
    df = df[fact_cols]
    
    try:
        df = df.sort_index(key=lambda x: [int(re.search(r'fail_(\d+)$', i).group(1)) for i in x])
    except (AttributeError, TypeError):
        df = df.sort_index()
    
    if fact_know_thresh is None or bind_thresh is None:
        return df, json_entities, evaluated_entities
    
    binding_failures = []
    for fail_name, row in df.iterrows():
        fact_columns = [c for c in df.columns if c not in BINDING_TYPES]
        
        fact_values = row[fact_columns].dropna()
        if len(fact_values) == 0:
            continue
        
        all_facts_known = all(v >= fact_know_thresh for v in fact_values)
        binding_fails = row.get("binding_raw", 1.0) < bind_thresh
        
        if all_facts_known and binding_fails:
            binding_failures.append(fail_name)
    
    return df, binding_failures, json_entities, evaluated_entities

# %% [markdown]
# ## Discover Available Models

# %%
models = sorted(os.listdir(BASE_DIR))
print(f"Found {len(models)} models:")
for m in models:
    print(f"  - {m}")

# %% [markdown]
# ## Evaluate All Models

# %%
all_results = {}
all_dfs = {}

for model_name in models:
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print('='*60)
    
    try:
        df, binding_failures, json_entities, evaluated_entities = count_binding_failures(
            model_name=model_name,
            temperature=TEMPERATURE,
            fact_know_thresh=FACT_KNOW_THRESH,
            bind_thresh=BIND_THRESH,
            base_dir=BASE_DIR,
            json_path=DATASET_JSON,
            filter_to_json_only=True  # Only include entities from JSON file
        )
        
        # Report on excluded entities (not in JSON)
        excluded_info = df.attrs.get('excluded_info', {'count': 0, 'dirs': []})
        if excluded_info['count'] > 0:
            print(f"⚠️  Excluded {excluded_info['count']} entities not in JSON file:")
            for dir_name in excluded_info['dirs'][:5]:
                print(f"      - {dir_name}")
            if excluded_info['count'] > 5:
                print(f"      ... and {excluded_info['count'] - 5} more")
            print()
        
        # Check if binding_raw column exists
        if 'binding_raw' not in df.columns:
            print(f"⚠️  No 'binding_raw' column found - model may not have completed binding evaluation")
            print(f"   Available columns: {list(df.columns)}")
            print(f"   Total entities in df: {len(df)}")
            all_results[model_name] = {'error': 'no_binding_raw_column', 'n_entities': len(df)}
            continue
        
        # Now df already only contains entities from JSON, so just filter for binding_raw
        df_filtered = df[df['binding_raw'].notna()]
        n_evaluated = len(df_filtered)
        
        if n_evaluated == 0:
            print(f"⚠️  No entities with binding_raw results")
            all_results[model_name] = {'error': 'no_binding_results', 'n_entities': len(df)}
            continue
        
        # Get the original "fail_*" names for filtered entries
        reverse_name_mapping = df.attrs.get('reverse_name_mapping', {})
        filtered_names = set()
        for display_name in df_filtered.index:
            original_name = reverse_name_mapping.get(display_name, display_name)
            filtered_names.add(original_name)
        
        # Compute set differences - should now show no entities outside JSON
        in_filtered_not_in_json = filtered_names - json_entities
        in_json_not_in_filtered = json_entities - filtered_names
        in_both = filtered_names & json_entities
        
        model_stats = {
            'n_evaluated': n_evaluated,
            'n_binding_failures': len(binding_failures),
            'n_in_json': len(json_entities),
            'n_excluded_not_in_json': excluded_info['count'],
            'n_in_filtered_not_in_json': len(in_filtered_not_in_json),
            'n_in_json_not_in_filtered': len(in_json_not_in_filtered),
            'n_in_both': len(in_both),
            'entities_excluded_not_in_json': excluded_info['dirs'],
        }
        
        # Compute stats for each binding type
        for b in BINDING_TYPES:
            if b in df_filtered.columns:
                n_failures = sum(df_filtered[b] < BIND_THRESH)
                pct_failures = (n_failures / n_evaluated * 100) if n_evaluated > 0 else 0
                mean_acc = df_filtered[b].mean()
                
                model_stats[f'{b}_n_failures'] = n_failures
                model_stats[f'{b}_pct_failures'] = pct_failures
                model_stats[f'{b}_mean_acc'] = mean_acc
        
        all_results[model_name] = model_stats
        all_dfs[model_name] = df_filtered
        
        print(f"Samples evaluated (in df_filtered): {n_evaluated}")
        print(f"Entities in JSON file: {len(json_entities)}")
        print(f"Entities excluded (not in JSON): {excluded_info['count']}")
        print(f"Binding failures (facts known, binding failed): {len(binding_failures)}")
        print()
        
        # Report on set differences (should now be empty for in_filtered_not_in_json)
        print(f"  📊 Entity Coverage:")
        print(f"    - In both df_filtered AND JSON: {len(in_both)}")
        print(f"    - In df_filtered but NOT in JSON: {len(in_filtered_not_in_json)} (should be 0)")
        print(f"    - In JSON but NOT in df_filtered: {len(in_json_not_in_filtered)}")
        
        if in_filtered_not_in_json:
            # This should not happen with filter_to_json_only=True
            print(f"\n  ❌ ERROR: Entities in df_filtered but NOT in JSON ({len(in_filtered_not_in_json)}):")
            for entity in sorted(in_filtered_not_in_json)[:10]:
                print(f"      - {entity}")
            if len(in_filtered_not_in_json) > 10:
                print(f"      ... and {len(in_filtered_not_in_json) - 10} more")
        
        print()
        for b in BINDING_TYPES:
            if b in df_filtered.columns:
                print(f"  {b}:")
                print(f"    - Failures: {model_stats[f'{b}_n_failures']} ({model_stats[f'{b}_pct_failures']:.1f}%)")
                print(f"    - Mean accuracy: {model_stats[f'{b}_mean_acc']:.3f}")
        
    except Exception as e:
        import traceback
        print(f"ERROR evaluating {model_name}: {e}")
        traceback.print_exc()
        all_results[model_name] = {'error': str(e)}

# %% [markdown]
# ## Summary Comparison Table

# %%
# Create comparison DataFrame
comparison_data = []

for model_name, stats in all_results.items():
    if 'error' in stats:
        continue
    
    row = {
        'model': model_name, 
        'n_evaluated': stats['n_evaluated'],
        'n_in_json': stats.get('n_in_json', 0),
        'n_excluded': stats.get('n_excluded_not_in_json', 0),
    }
    
    for b in BINDING_TYPES:
        if f'{b}_mean_acc' in stats:
            row[f'{b}_acc'] = stats[f'{b}_mean_acc']
            row[f'{b}_fail%'] = stats[f'{b}_pct_failures']
    
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index('model')

print("\n" + "="*80)
print("SUMMARY: Mean Binding Accuracy by Model")
print("="*80)

# Display accuracy columns
acc_cols = [c for c in comparison_df.columns if c.endswith('_acc')]
print(comparison_df[['n_evaluated', 'n_in_json', 'n_excluded'] + acc_cols].round(3).to_string())

print("\n" + "="*80)
print("SUMMARY: Binding Failure Rate (%) by Model")
print("="*80)

# Display failure percentage columns
fail_cols = [c for c in comparison_df.columns if c.endswith('_fail%')]
print(comparison_df[['n_evaluated', 'n_excluded'] + fail_cols].round(1).to_string())

# %% [markdown]
# ## Visualization

# %%
import matplotlib.pyplot as plt

# Plot mean accuracy comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
acc_df = comparison_df[acc_cols].T
acc_df.index = [c.replace('_acc', '') for c in acc_df.index]
acc_df.plot(kind='bar', ax=axes[0], rot=45)
axes[0].set_title('Mean Binding Accuracy by Model')
axes[0].set_xlabel('Binding Type')
axes[0].set_ylabel('Accuracy')
axes[0].legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[0].set_ylim(0, 1)
axes[0].axhline(y=BIND_THRESH, color='r', linestyle='--', alpha=0.5, label=f'Threshold ({BIND_THRESH})')

# Failure rate plot
fail_df = comparison_df[fail_cols].T
fail_df.index = [c.replace('_fail%', '') for c in fail_df.index]
fail_df.plot(kind='bar', ax=axes[1], rot=45)
axes[1].set_title('Binding Failure Rate (%) by Model')
axes[1].set_xlabel('Binding Type')
axes[1].set_ylabel('Failure Rate (%)')
axes[1].legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.savefig(f'binding_comparison_{TYPE}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to: binding_comparison_{TYPE}.png")

# %% [markdown]
# ## Detailed Per-Model Statistics

# %%
for model_name, df in all_dfs.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print('='*60)
    print(f"Shape: {df.shape}")
    print(f"\nBinding columns statistics:")
    print(df[BINDING_TYPES].describe().round(3))

# %% [markdown]
# ## Export Results

# %%
# Save comparison table to CSV
comparison_df.to_csv(f'binding_comparison_{TYPE}_results.csv')
print(f"Results saved to: binding_comparison_{TYPE}_results.csv")

# Save full results dict to JSON
with open(f'results/binding_all_results_{TYPE}.json', 'w') as f:
    # Convert to serializable format
    serializable_results = {}
    for k, v in all_results.items():
        serializable_results[k] = {}
        for kk, vv in v.items():
            if isinstance(vv, (float, int)):
                serializable_results[k][kk] = float(vv)
            elif isinstance(vv, (list, set)):
                serializable_results[k][kk] = list(vv)
            else:
                serializable_results[k][kk] = vv
    json.dump(serializable_results, f, indent=2)
print(f"Full results saved to: binding_all_results_{TYPE}.json")

### Custom

In [ ]:
"""
Binding Failures Evaluation - All Models Comparison
====================================================
"""

import os
import json
import glob
import re
from pathlib import Path
import pandas as pd
from typing import Dict, List, Tuple, Set


TYPE = 'custom'
BASE_DIR = f"./binding_failures_{TYPE}_temp0.3"
DATASET_JSON = f'binding_test_cases_{TYPE}.json'
TEMPERATURE = 0.3
FACT_KNOW_THRESH = 0.6
BIND_THRESH = 0.6 # Correct when 60% of eval_bind are correct. Meaning that there are 

BINDING_TYPES = [
    "binding_raw",
    "binding_reason", 
    "binding_no_reason",
    "binding_mc_raw",
    "binding_mc_reason",
    "binding_mc_no_reason",
]

# %%
def _compute_accuracy(eval_dir: Path, pattern: str = "eval_*.json") -> float:
    correct, total = 0, 0
    for f in eval_dir.glob(pattern):
        if f.name.startswith("_debug"):
            continue
        data = json.load(open(f))
        total += 1
        is_correct = data.get('correct', False) or data.get('evaluation', {}).get('correct', False)
        if is_correct:
            correct += 1
    return correct / total if total > 0 else float('nan')


def count_binding_failures(model_name: str, temperature: float = 0.3, 
                           base_dir: str = "experiments/Binding_Failures/binding_failures",
                           json_path: str = None,
                           fact_know_thresh: float = None, 
                           bind_thresh: float = None,
                           show_entity_names: bool = False,
                           filter_to_json_only: bool = True):
    """
    Build a table of fact/binding accuracies per fail, and optionally identify binding failures.
    
    Args:
        filter_to_json_only: If True, only include entities that exist in the JSON file.
    
    Returns:
        If fact_know_thresh and bind_thresh are None:
            df, json_entities, evaluated_entities
        Otherwise:
            df, binding_failures, json_entities, evaluated_entities
    
    Note: json_entities are in "fail_{name}" format to match directory names.
    """
    model_path = Path(base_dir) / model_name.replace(':', '_')
    
    if not model_path.exists():
        raise FileNotFoundError(f"Model path not found: {model_path}")
    
    # Load JSON entities for comparison
    # Entity names in JSON are converted to directory format: fail_{entity.replace(' ', '_').lower()}
    json_entities: Set[str] = set()  # Will store full "fail_*" names
    json_entity_parts: Set[str] = set()  # Will store just the entity part (without "fail_" prefix)
    entity_to_id = {}
    if json_path and os.path.exists(json_path):
        with open(json_path, 'r') as f:
            data = json.load(f)
        for idx, entity_name in enumerate(data['test_cases'].keys(), start=1):
            # Convert: "Some Entity Name" -> "some_entity_name"
            dir_name = entity_name.replace(' ', '_').lower()
            entity_to_id[dir_name] = idx
            json_entity_parts.add(dir_name)
            json_entities.add(f"fail_{dir_name}")  # Full directory name format
    
    results = {}
    fail_dirs = list(model_path.glob("fail_*"))
    
    # Track evaluated entities (the raw entity part, not the display name)
    evaluated_entities: Set[str] = set()
    
    name_mapping = {}
    entity_part_mapping = {}  # Maps display_name back to entity_part
    
    # Track which directories are excluded because they're not in JSON
    excluded_dirs = []
    
    for fail_dir in fail_dirs:
        original_name = fail_dir.name
        entity_part = original_name[5:]  # Remove "fail_" prefix
        
        # Skip entities not in JSON if filtering is enabled
        if filter_to_json_only and json_path and original_name not in json_entities:
            excluded_dirs.append(original_name)
            continue
        
        evaluated_entities.add(entity_part)
        
        if json_path and entity_part in entity_to_id:
            display_name = f"fail_{entity_to_id[entity_part]}" if not show_entity_names else original_name
        elif re.match(r'^\d+$', entity_part):
            display_name = original_name
        else:
            display_name = original_name if show_entity_names else original_name
        
        name_mapping[original_name] = display_name
        entity_part_mapping[display_name] = entity_part
    
    # Store excluded directories info for later reporting
    excluded_info = {
        'count': len(excluded_dirs),
        'dirs': excluded_dirs
    }
    
    for fail_dir in fail_dirs:
        original_name = fail_dir.name
        
        # Skip if not in name_mapping (means it was excluded)
        if original_name not in name_mapping:
            continue
            
        display_name = name_mapping[original_name]
        results[display_name] = {}
        
        facts = set()
        for subdir in fail_dir.iterdir():
            if subdir.is_dir() and subdir.name.startswith("eval_"):
                name = subdir.name
                if re.match(r'^eval_[a-z]$', name):
                    facts.add(name[5:])
                elif re.match(r'^eval_\w+_[a-z]$', name):
                    facts.add(name.split('_')[-1])
        
        facts = sorted(facts)
        
        for fact in facts:
            for eval_dir in fail_dir.glob(f"eval_*_{fact}"):
                qtype = eval_dir.name.replace(f"_{fact}", "").replace("eval_", "")
                results[display_name][f"{qtype}_{fact}"] = _compute_accuracy(eval_dir, pattern="eval_*.json")
            
            yesno_dir = fail_dir / f"eval_{fact}"
            if yesno_dir.exists():
                results[display_name][f"yesno_{fact}"] = _compute_accuracy(yesno_dir, pattern="eval_*.json")
        
        for binding_type in BINDING_TYPES:
            binding_dir = fail_dir / f"eval_{binding_type}"
            if binding_dir.exists():
                results[display_name][binding_type] = _compute_accuracy(
                    binding_dir, pattern="eval_*.json"
                )
    
    df = pd.DataFrame.from_dict(results, orient='index')
    
    # Store the entity_part_mapping in df attrs for later use
    # Also store reverse mapping: display_name -> original "fail_*" name
    df.attrs['entity_part_mapping'] = entity_part_mapping
    df.attrs['name_mapping'] = name_mapping  # original_name -> display_name
    df.attrs['reverse_name_mapping'] = {v: k for k, v in name_mapping.items()}  # display_name -> original_name
    df.attrs['excluded_info'] = excluded_info  # Info about excluded directories
    
    fact_cols = []
    all_facts = set()
    for col in df.columns:
        if col not in BINDING_TYPES:
            match = re.search(r'_([a-z])$', col)
            if match:
                all_facts.add(match.group(1))
    
    all_prefixes = sorted(set(col.rsplit('_', 1)[0] for col in df.columns if col not in BINDING_TYPES))
    for fact in sorted(all_facts):
        for prefix in all_prefixes:
            col = f"{prefix}_{fact}"
            if col in df.columns:
                fact_cols.append(col)
    
    for binding_type in BINDING_TYPES:
        if binding_type in df.columns:
            fact_cols.append(binding_type)
    
    df = df[fact_cols]
    
    try:
        df = df.sort_index(key=lambda x: [int(re.search(r'fail_(\d+)$', i).group(1)) for i in x])
    except (AttributeError, TypeError):
        df = df.sort_index()
    
    if fact_know_thresh is None or bind_thresh is None:
        return df, json_entities, evaluated_entities
    
    binding_failures = []
    for fail_name, row in df.iterrows():
        fact_columns = [c for c in df.columns if c not in BINDING_TYPES]
        
        fact_values = row[fact_columns].dropna()
        if len(fact_values) == 0:
            continue
        
        all_facts_known = all(v >= fact_know_thresh for v in fact_values)
        binding_fails = row.get("binding_raw", 1.0) < bind_thresh
        
        if all_facts_known and binding_fails:
            binding_failures.append(fail_name)
    
    return df, binding_failures, json_entities, evaluated_entities

# %% [markdown]
# ## Discover Available Models

# %%
models = sorted(os.listdir(BASE_DIR))
print(f"Found {len(models)} models:")
for m in models:
    print(f"  - {m}")

# %% [markdown]
# ## Evaluate All Models

# %%
all_results = {}
all_dfs = {}

for model_name in models:
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print('='*60)
    
    try:
        df, binding_failures, json_entities, evaluated_entities = count_binding_failures(
            model_name=model_name,
            temperature=TEMPERATURE,
            fact_know_thresh=FACT_KNOW_THRESH,
            bind_thresh=BIND_THRESH,
            base_dir=BASE_DIR,
            json_path=DATASET_JSON,
            filter_to_json_only=True  # Only include entities from JSON file
        )
        
        # Report on excluded entities (not in JSON)
        excluded_info = df.attrs.get('excluded_info', {'count': 0, 'dirs': []})
        if excluded_info['count'] > 0:
            print(f"⚠️  Excluded {excluded_info['count']} entities not in JSON file:")
            for dir_name in excluded_info['dirs'][:5]:
                print(f"      - {dir_name}")
            if excluded_info['count'] > 5:
                print(f"      ... and {excluded_info['count'] - 5} more")
            print()
        
        # Check if binding_raw column exists
        if 'binding_raw' not in df.columns:
            print(f"⚠️  No 'binding_raw' column found - model may not have completed binding evaluation")
            print(f"   Available columns: {list(df.columns)}")
            print(f"   Total entities in df: {len(df)}")
            all_results[model_name] = {'error': 'no_binding_raw_column', 'n_entities': len(df)}
            continue
        
        # Now df already only contains entities from JSON, so just filter for binding_raw
        df_filtered = df[df['binding_raw'].notna()]
        n_evaluated = len(df_filtered)
        
        if n_evaluated == 0:
            print(f"⚠️  No entities with binding_raw results")
            all_results[model_name] = {'error': 'no_binding_results', 'n_entities': len(df)}
            continue
        
        # Get the original "fail_*" names for filtered entries
        reverse_name_mapping = df.attrs.get('reverse_name_mapping', {})
        filtered_names = set()
        for display_name in df_filtered.index:
            original_name = reverse_name_mapping.get(display_name, display_name)
            filtered_names.add(original_name)
        
        # Compute set differences - should now show no entities outside JSON
        in_filtered_not_in_json = filtered_names - json_entities
        in_json_not_in_filtered = json_entities - filtered_names
        in_both = filtered_names & json_entities
        
        model_stats = {
            'n_evaluated': n_evaluated,
            'n_binding_failures': len(binding_failures),
            'n_in_json': len(json_entities),
            'n_excluded_not_in_json': excluded_info['count'],
            'n_in_filtered_not_in_json': len(in_filtered_not_in_json),
            'n_in_json_not_in_filtered': len(in_json_not_in_filtered),
            'n_in_both': len(in_both),
            'entities_excluded_not_in_json': excluded_info['dirs'],
        }
        
        # Compute stats for each binding type
        for b in BINDING_TYPES:
            if b in df_filtered.columns:
                n_failures = sum(df_filtered[b] < BIND_THRESH)
                pct_failures = (n_failures / n_evaluated * 100) if n_evaluated > 0 else 0
                mean_acc = df_filtered[b].mean()
                
                model_stats[f'{b}_n_failures'] = n_failures
                model_stats[f'{b}_pct_failures'] = pct_failures
                model_stats[f'{b}_mean_acc'] = mean_acc
        
        all_results[model_name] = model_stats
        all_dfs[model_name] = df_filtered
        
        print(f"Samples evaluated (in df_filtered): {n_evaluated}")
        print(f"Entities in JSON file: {len(json_entities)}")
        print(f"Entities excluded (not in JSON): {excluded_info['count']}")
        print(f"Binding failures (facts known, binding failed): {len(binding_failures)}")
        print()
        
        # Report on set differences (should now be empty for in_filtered_not_in_json)
        print(f"  📊 Entity Coverage:")
        print(f"    - In both df_filtered AND JSON: {len(in_both)}")
        print(f"    - In df_filtered but NOT in JSON: {len(in_filtered_not_in_json)} (should be 0)")
        print(f"    - In JSON but NOT in df_filtered: {len(in_json_not_in_filtered)}")
        
        if in_filtered_not_in_json:
            # This should not happen with filter_to_json_only=True
            print(f"\n  ❌ ERROR: Entities in df_filtered but NOT in JSON ({len(in_filtered_not_in_json)}):")
            for entity in sorted(in_filtered_not_in_json)[:10]:
                print(f"      - {entity}")
            if len(in_filtered_not_in_json) > 10:
                print(f"      ... and {len(in_filtered_not_in_json) - 10} more")
        
        print()
        for b in BINDING_TYPES:
            if b in df_filtered.columns:
                print(f"  {b}:")
                print(f"    - Failures: {model_stats[f'{b}_n_failures']} ({model_stats[f'{b}_pct_failures']:.1f}%)")
                print(f"    - Mean accuracy: {model_stats[f'{b}_mean_acc']:.3f}")
        
    except Exception as e:
        import traceback
        print(f"ERROR evaluating {model_name}: {e}")
        traceback.print_exc()
        all_results[model_name] = {'error': str(e)}

# %% [markdown]
# ## Summary Comparison Table

# %%
# Create comparison DataFrame
comparison_data = []

for model_name, stats in all_results.items():
    if 'error' in stats:
        continue
    
    row = {
        'model': model_name, 
        'n_evaluated': stats['n_evaluated'],
        'n_in_json': stats.get('n_in_json', 0),
        'n_excluded': stats.get('n_excluded_not_in_json', 0),
    }
    
    for b in BINDING_TYPES:
        if f'{b}_mean_acc' in stats:
            row[f'{b}_acc'] = stats[f'{b}_mean_acc']
            row[f'{b}_fail%'] = stats[f'{b}_pct_failures']
    
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index('model')

print("\n" + "="*80)
print("SUMMARY: Mean Binding Accuracy by Model")
print("="*80)

# Display accuracy columns
acc_cols = [c for c in comparison_df.columns if c.endswith('_acc')]
print(comparison_df[['n_evaluated', 'n_in_json', 'n_excluded'] + acc_cols].round(3).to_string())

print("\n" + "="*80)
print("SUMMARY: Binding Failure Rate (%) by Model")
print("="*80)

# Display failure percentage columns
fail_cols = [c for c in comparison_df.columns if c.endswith('_fail%')]
print(comparison_df[['n_evaluated', 'n_excluded'] + fail_cols].round(1).to_string())

# %% [markdown]
# ## Visualization

# %%
import matplotlib.pyplot as plt

# Plot mean accuracy comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
acc_df = comparison_df[acc_cols].T
acc_df.index = [c.replace('_acc', '') for c in acc_df.index]
acc_df.plot(kind='bar', ax=axes[0], rot=45)
axes[0].set_title('Mean Binding Accuracy by Model')
axes[0].set_xlabel('Binding Type')
axes[0].set_ylabel('Accuracy')
axes[0].legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[0].set_ylim(0, 1)
axes[0].axhline(y=BIND_THRESH, color='r', linestyle='--', alpha=0.5, label=f'Threshold ({BIND_THRESH})')

# Failure rate plot
fail_df = comparison_df[fail_cols].T
fail_df.index = [c.replace('_fail%', '') for c in fail_df.index]
fail_df.plot(kind='bar', ax=axes[1], rot=45)
axes[1].set_title('Binding Failure Rate (%) by Model')
axes[1].set_xlabel('Binding Type')
axes[1].set_ylabel('Failure Rate (%)')
axes[1].legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.savefig(f'binding_comparison_{TYPE}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to: binding_comparison_{TYPE}.png")

# %% [markdown]
# ## Detailed Per-Model Statistics

# %%
for model_name, df in all_dfs.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print('='*60)
    print(f"Shape: {df.shape}")
    print(f"\nBinding columns statistics:")
    print(df[BINDING_TYPES].describe().round(3))

# %% [markdown]
# ## Export Results

# %%
# Save comparison table to CSV
comparison_df.to_csv(f'binding_comparison_{TYPE}_results.csv')
print(f"Results saved to: binding_comparison_{TYPE}_results.csv")

# Save full results dict to JSON
with open(f'results/binding_all_results_{TYPE}.json', 'w') as f:
    # Convert to serializable format
    serializable_results = {}
    for k, v in all_results.items():
        serializable_results[k] = {}
        for kk, vv in v.items():
            if isinstance(vv, (float, int)):
                serializable_results[k][kk] = float(vv)
            elif isinstance(vv, (list, set)):
                serializable_results[k][kk] = list(vv)
            else:
                serializable_results[k][kk] = vv
    json.dump(serializable_results, f, indent=2)
print(f"Full results saved to: binding_all_results_{TYPE}.json")

# Plots Heatmap

In [ ]:
# there are 2 JSON datasets in the results folder: binding_all_results + football_players
import os
import json
import glob
import re
from pathlib import Path
import pandas as pd
from typing import Dict, List, Tuple, Set
# there are 2 JSON datasets in the results folder: binding_all_results + football_players


data_football = None
data_custom = None

data = {}

with open('results/binding_all_results_custom.json', 'r') as fp:
    data_custom = json.load(fp)

with open('results/binding_all_results_athletes.json', 'r') as fp:
    data_athletes = json.load(fp)


for k, v in data_custom.items():
    data[f"custom_{k}"] = v

for k, v in data_athletes.items():
    data[f"athletes_{k}"] = v


### Main paper heatmap

In [ ]:

# flatten JSON, keep only comparable models
rows = []
for model, stats in data.items():
    if "binding_raw_n_failures" not in stats:
        continue
    rows.append({
        "model": model,
        "n_known": stats["n_evaluated"],
        "binding_raw": stats["binding_raw_n_failures"],
        "binding_reason": stats["binding_reason_n_failures"],
        "binding_no_reason": stats["binding_no_reason_n_failures"],
        "binding_mc_raw": stats["binding_mc_raw_n_failures"],
        "binding_mc_reason": stats["binding_mc_reason_n_failures"],
        "binding_mc_no_reason": stats["binding_mc_no_reason_n_failures"],
    })

df = pd.DataFrame(rows).set_index("model")
df

In [ ]:
import pandas as pd

# df is your dataframe with index = model
# columns as shown in the table

# helper to extract (failures, total)
def row_to_triplet(row):
    total = int(row["n_known"])
    return [
        (int(row["binding_raw"]), total),
        (int(row["binding_reason"]), total),
        (int(row["binding_mc_raw"]), total),
    ]

# model name mapping: (football_row, custom_row)
models = [
    ("athletes_claude-opus-4-5-20251101", "custom_claude-opus-4-5-20251101"),
    ("athletes_claude-sonnet-4-20250514", "custom_claude-sonnet-4-20250514"),
    ("athletes_gpt-4o-2024-08-06", "custom_gpt-4o-2024-08-06"),
    ("athletes_gemini-2.0-flash", "custom_gemini-2.0-flash"),
    ("athletes_deepseek-chat", "custom_deepseek-chat"),
]

data = []

for athletes, custom in models:
    fb = df.loc[athletes]
    cu = df.loc[custom]

    entry = (
        row_to_triplet(fb) +
        row_to_triplet(cu)
    )
    data.append(entry)

# data now matches the target list structure
print(data)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib
# matplotlib.use('Qt5Agg')  # or 'Qt5Agg' depending on your system

models = [
    "Claude Opus 4.5",
    "Claude Sonnet 4",
    "GPT-4o",
    "Gemini 2.0 Flash",
    "DeepSeek-V3",
]

# Column labels - MC replaced with Multi-choice
columns = [
    'Instant',
    'Reasoning', 
    'MCQ Instant',
    'Instant',
    'Reasoning',
    'MCQ Instant'
]

# Create figure with larger size for better visibility
fig, ax = plt.subplots(figsize=(14, 9))

# Calculate percentages for colormap
percentages = np.full((len(models), len(columns)), np.nan)
for i in range(len(models)):
    for j in range(len(columns)):
        if data[i][j] is not None:
            failures, total = data[i][j]
            percentages[i, j] = (failures / total * 100) if total > 0 else 0

# Use viridis colormap (colorblind-friendly)
norm = Normalize(vmin=0, vmax=100)
cmap = plt.cm.viridis_r

# Create heatmap
im = ax.imshow(percentages, cmap=cmap, norm=norm, interpolation='nearest', aspect='auto')

# Set ticks and labels
ax.set_xticks(np.arange(len(columns)))
ax.set_xticklabels(columns, fontsize=30, rotation=45, ha='right')

# Y-axis labels with 45 degree rotation
ax.set_yticks(np.arange(len(models)))
ax.set_yticklabels(models, fontsize=30, rotation=45, ha='right')

# Add vertical separator line between Athletes and Custom
ax.axvline(x=2.5, color='black', linewidth=3, linestyle='-')

# Add colorbar with larger font
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Failure Rate (%)', fontsize=30)
cbar.ax.tick_params(labelsize=21)

# Add value annotations - percentage + known facts in parentheses
for i in range(len(models)):
    for j in range(len(columns)):
        if data[i][j] is None:
            ax.text(j, i, 'N/A', ha='center', va='center',
                    color='gray', fontsize=14, style='italic')
        else:
            failures, total = data[i][j]
            percentage = (failures / total * 100) if total > 0 else 0

            # Determine text color based on background luminance
            r, g, b, _ = cmap(norm(percentage))
            luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
            text_color = 'white' if luminance < 0.4 else 'black'

            # Two separate texts with different font sizes
            ax.text(j, i-0.12, f'{percentage:.0f}%', ha='center', va='center',
                    color=text_color, fontsize=30)
            ax.text(j, i+0.22, f'(N={int(total)})', ha='center', va='center',
                    color=text_color, fontsize=21)


# Style the plot
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks(np.arange(len(columns) + 1) - .5, minor=True)
ax.set_yticks(np.arange(len(models) + 1) - .5, minor=True)
ax.grid(False)

# Add cell borders
for i in range(len(models)):
    for j in range(len(columns)):
        ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, 
                                   fill=False, lw=1.5, edgecolor='white'))
        
# Add a marker inside specific cells
for i in [0, 1]:  # Claude rows
    for j in [1, 4]:  # Reasoning columns
        ax.text(j + 0.36, i - 0.30, '★', fontsize=30, ha='center', va='center', 
                color='black')

# Add dataset labels at top
ax.text(1, -0.90, 'Athletes Dataset', ha='center', va='center',
       fontsize=30)
ax.text(4, -0.90, 'Custom Dataset', ha='center', va='center',
       fontsize=30)

plt.tight_layout()

# Save as PDF (vector format for publications)
plt.savefig('binding_failures_merged_heatmap.pdf', format='pdf', bbox_inches='tight', facecolor='white', pad_inches=0.3)
print('✓ Publication-ready heatmap saved as PDF')
plt.show()

### Appendix heatmap

In [ ]:
import pandas as pd

# df is your dataframe with index = model
# columns as shown in the table

# helper to extract (failures, total)
def row_to_triplet(row):
    total = int(row["n_known"])
    return [
        (int(row["binding_raw"]), total),
        (int(row["binding_reason"]), total),
        (int(row["binding_no_reason"]), total),
        (int(row["binding_mc_raw"]), total),
        (int(row["binding_mc_reason"]), total),
        (int(row["binding_mc_no_reason"]), total),

    ]

# model name mapping: (football_row, custom_row)
models = [
    ("athletes_claude-opus-4-5-20251101", "custom_claude-opus-4-5-20251101"),
    ("athletes_claude-sonnet-4-20250514", "custom_claude-sonnet-4-20250514"),
    ("athletes_gpt-4o-2024-08-06", "custom_gpt-4o-2024-08-06"),
    ("athletes_gemini-2.0-flash", "custom_gemini-2.0-flash"),
    ("athletes_deepseek-chat", "custom_deepseek-chat"),
]

data = []

for athletes, custom in models:
    fb = df.loc[athletes]
    cu = df.loc[custom]

    entry = (
        row_to_triplet(fb) +
        row_to_triplet(cu)
    )
    data.append(entry)

# data now matches the target list structure
print(data)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

models = [
    "Claude Opus 4.5",
    "Claude Sonnet 4",
    "GPT-4o",
    "Gemini 2.0 Flash",
    "DeepSeek-V3",
]

# Column labels - no bold
columns = [
    'Instant',
    'Reasoning', 
    'No Reasoning', 
    'MCQ Instant',
    'MCQ Reasoning',
    'MCQ No Reasoning',

    'Instant',
    'Reasoning', 
    'No Reasoning', 
    'MCQ Instant',
    'MCQ Reasoning',
    'MCQ No Reasoning',
]

# Create figure with larger size for better visibility
fig, ax = plt.subplots(figsize=(28, 9))

# Calculate percentages for colormap
percentages = np.full((len(models), len(columns)), np.nan)
for i in range(len(models)):
    for j in range(len(columns)):
        if data[i][j] is not None:
            failures, total = data[i][j]
            percentages[i, j] = (failures / total * 100) if total > 0 else 0

# Use viridis colormap (colorblind-friendly)
norm = Normalize(vmin=0, vmax=100)
cmap = plt.cm.viridis_r
# cmap = plt.cm.cividis_r
# cmap = plt.cm.inferno_r
# cmap = plt.cm.jet

# Create heatmap
im = ax.imshow(percentages, cmap=cmap, norm=norm, interpolation='nearest', aspect='auto')

# Set ticks and labels
ax.set_xticks(np.arange(len(columns)))
ax.set_xticklabels(columns, fontsize=30, rotation=45, ha='right')

# Y-axis labels with 45 degree rotation
ax.set_yticks(np.arange(len(models)))
ax.set_yticklabels(models, fontsize=30, rotation=45, ha='right')

# Add vertical separator line between Athletes and Custom
ax.axvline(x=5.5, color='black', linewidth=3, linestyle='-')

# Add colorbar with larger font
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Failure Rate (%)', fontsize=30)
cbar.ax.tick_params(labelsize=21)

# Add value annotations - percentage + known facts in parentheses
for i in range(len(models)):
    for j in range(len(columns)):
        if data[i][j] is None:
            ax.text(j, i, 'N/A', ha='center', va='center',
                    color='gray', fontsize=14, style='italic')
        else:
            failures, total = data[i][j]
            percentage = (failures / total * 100) if total > 0 else 0

            # Determine text color based on background luminance
            r, g, b, _ = cmap(norm(percentage))
            luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
            text_color = 'white' if luminance < 0.4 else 'black'

            # Two separate texts with different font sizes
            ax.text(j, i-0.12, f'{percentage:.0f}%', ha='center', va='center',
                    color=text_color, fontsize=30)
            ax.text(j, i+0.22, f'(N={int(total)})', ha='center', va='center',
                    color=text_color, fontsize=21)


# Style the plot
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks(np.arange(len(columns) + 1) - .5, minor=True)
ax.set_yticks(np.arange(len(models) + 1) - .5, minor=True)
ax.grid(False)

# Add cell borders
for i in range(len(models)):
    for j in range(len(columns)):
        ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, 
                                   fill=False, lw=1.5, edgecolor='white'))

# Add a marker inside specific cells
for i in [0, 1]:  # Claude rows
    for j in [1, 4, 7, 10]:  # Reasoning columns
        ax.text(j + 0.36, i - 0.25, '★', fontsize=30, ha='center', va='center', 
                color='black')

# Add dataset labels at top
ax.text(2.5, -0.90, 'Athletes Dataset', ha='center', va='center',
       fontsize=30)
ax.text(8.5, -0.90, 'Custom Dataset', ha='center', va='center',
       fontsize=30)

plt.tight_layout()
plt.savefig('binding_failures_merged_heatmap_appendix.pdf', format='pdf', bbox_inches='tight', facecolor='white', pad_inches=0.3)
print('✓ Publication-ready heatmap saved to: binding_failures_merged_heatmap_appendix.pdf')
plt.show()